In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

data = {
    'name':       ['Alice','Bob','Carol','Dave','Eve','Frank','Grace','Hank','Iris','Jack',
                   'Karen','Leo','Mia','Ned','Olivia','Pete','Quinn','Rose','Sam','Tina'],
    'dept':       ['Sales','Tech','Sales','HR','Tech','Sales','Tech','HR','Sales','Tech',
                   'HR','Sales','Tech','HR','Sales','Tech','Sales','HR','Tech','Sales'],
    'age':        [25, 32, 28, 45, 36, 52, 29, 41, 33, 38, 27, 60, 31, 44, 26, 35, 48, 39, 30, 55],
    'experience': [ 2,  8,  4, 20, 12, 28,  5, 18,  9, 14,  3, 35,  7, 21,  1, 11, 24, 16,  6, 30],
    'salary':     [42, 85, 47, 61, 90, 52, 88, 63, 45, 91, 58, 54, 86, 65, 44, 89, 50, 67, 82, 200],  # 200 = outlier
    'rating':     [ 4,  5,  3,  4,  5,  3,  5,  4,  4,  5,  3,  4,  5,  3,  4,  5,  4,  3,  5,  4],
    'sales_calls':[ 40,  0, 38,  0,  0, 45,  0,  0, 42,  0,  0, 35,  0,  0, 39,  0, 44,  0,  0, 41],
    'deals_won':  [ 12,  0, 10,  0,  0, 15,  0,  0, 11,  0,  0,  9,  0,  0, 13,  0, 14,  0,  0, 12],
    'join_date':  ['2022-03-15','2016-07-01','2020-11-20','2003-05-10','2012-01-25',
                   '1996-08-14','2019-04-03','2006-02-28','2015-09-17','2010-06-12',
                   '2021-12-01','1989-03-22','2017-08-30','2003-11-05','2023-01-10',
                   '2013-07-19','2000-04-27','2008-10-03','2018-02-14','1994-06-30'],
}

df = pd.DataFrame(data)
df['join_date'] = pd.to_datetime(df['join_date'])
# salary in thousands; age/experience in years; rating 1-5; sales_calls & deals_won for Sales dept only
df.head()

# Univariate Analysis

Examine **one variable at a time** — its distribution, shape, central tendency, spread, and outliers.

---

## Numerical vs Categorical

| Variable type | Question | Tools |
|---------------|----------|-------|
| **Numerical** | How is it distributed? Any outliers? | histogram, box plot, describe() |
| **Categorical** | Which category dominates? How many unique values? | count plot, value_counts() |

---

## Numerical Variables

### Step 1 — Summary stats

In [ ]:
df['col'].describe()       # count, mean, std, min, Q1, Q2, Q3, max
df['col'].skew()           # positive = right-skewed, negative = left-skewed

### Step 2 — Visualise distribution

In [ ]:
sns.histplot(data=df, x='col', bins=10)
sns.boxplot(data=df, x='col')

### Step 3 — Overlay mean/median to check skew

In [ ]:
plt.vlines(x=df['col'].mean(),   ymin=0, ymax=10, colors='blue', label='Mean')
plt.vlines(x=df['col'].median(), ymin=0, ymax=10, colors='red',  label='Median')
plt.legend()

- mean >> median → right-skewed (outliers on the high end)
- mean << median → left-skewed
- mean ≈ median → roughly symmetric

### Outlier detection — IQR method

In [ ]:
Q1  = df['col'].quantile(0.25)
Q3  = df['col'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['col'] < lower) | (df['col'] > upper)]

Box plot whiskers extend to ±1.5×IQR — points beyond are plotted as individual dots = outliers.

---

## Categorical Variables

### Summary

In [ ]:
df['col'].value_counts()           # frequency of each category, sorted descending
df['col'].value_counts(normalize=True)  # as proportions
df['col'].nunique()                # number of distinct categories
df.describe(include=[object])      # count, unique, top (mode), freq

### Visualise

In [ ]:
sns.countplot(data=df, x='col')

---

## Handling Messy Data Before Univariate

Real datasets often have dirty values in numeric columns (e.g. Tendulkar ODI: `'DNB'`, `'82*'`).

In [ ]:
# Check what non-numeric values exist
df[~df['col'].astype(str).str.match(r'^\d+$')]['col'].unique()

# Convert — coerce non-numeric to NaN, then fill
df['col'] = pd.to_numeric(df['col'], errors='coerce').fillna(0).astype(int)

---

## Side-by-Side Subplots for Multiple Variables

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 6))

plt.subplot(3, 2, 1)
sns.histplot(data=df, x='col1', ...)

plt.subplot(3, 2, 2)
sns.boxplot(data=df, x='col1', ...)

plt.tight_layout()

---

## Interpreting Results

| Observation | What it means |
|-------------|--------------|
| mean >> median | Right-skewed; outliers on high end pulling mean up |
| Wide IQR | High variability in middle 50% |
| Many outlier dots on box plot | Consider IQR-based removal or investigation |
| One category >> others in count plot | Data is imbalanced — may need stratified sampling |